## RAG functions
Shared retrieval and answering functions for Lab 12.
Callers install `databricks-ai-search` and `openai`, create the widgets and then `%run ./rag_functions`.

In [0]:
%run ./00_config

In [0]:
from databricks.ai_search.client import AISearchClient

ai_cfg = config["ai_search"]

client = AISearchClient(disable_notice=True)
index = client.get_index(endpoint_name=ai_cfg["endpoint_name"], index_name=index_name)

In [0]:
def embed_question(question: str) -> list[float]:
    """Embed the question with the same model that was used for the chunks."""
    row = spark.sql(
        f"SELECT ai_query('{ai_cfg['embedding_endpoint']}', :question) AS embedding",
        args={"question": question},
    ).first()
    return list(row["embedding"])

In [0]:
RESULT_COLUMNS = ["chunk_id", "video_id", "title", "channel_title", "category_name", "chunk_text"]


def retrieve(question: str, top_k: int | None = None, filters: dict | None = None) -> list[dict]:
    """Return the top-k chunks closest to the question, with metadata and score."""
    response = index.similarity_search(
        query_vector=embed_question(question),
        columns=RESULT_COLUMNS,
        num_results=top_k or ai_cfg["top_k"],
        filters=filters,
    )
    columns = [column["name"] for column in response["manifest"]["columns"]]
    rows = response.get("result", {}).get("data_array", [])
    return [dict(zip(columns, row)) for row in rows]

In [0]:
from databricks.sdk import WorkspaceClient
from openai import OpenAI

llm_cfg = config["llm"]

workspace = WorkspaceClient()
llm_client = OpenAI(
    api_key=workspace.config.authenticate()["Authorization"].split(" ", 1)[1],
    base_url=f"{workspace.config.host}/{llm_cfg['base_path']}",
)

In [0]:
period = config["source"]["trending_period"]

SYSTEM_PROMPT_RAG = f"""You answer questions about US YouTube trending videos from {period['start']} to {period['end']}.
Use ONLY the numbered context below. Do not use any other knowledge.
If the context does not contain the answer, reply exactly: "I don't know based on the available videos."
Cite the sources you used as [1], [2] using the context numbers.
Never invent video titles, channel names or numbers."""

SYSTEM_PROMPT_NO_RAG = "You answer questions about US YouTube trending videos. Answer briefly."


def format_context(chunks: list[dict]) -> str:
    return "\n\n".join(
        f"[{number}] {chunk['title']} ({chunk['channel_title']})\n{chunk['chunk_text']}"
        for number, chunk in enumerate(chunks, start=1)
    )


def build_messages(question: str, chunks: list[dict] | None) -> list[dict]:
    if chunks is None:
        return [
            {"role": "system", "content": SYSTEM_PROMPT_NO_RAG},
            {"role": "user", "content": question},
        ]
    return [
        {"role": "system", "content": SYSTEM_PROMPT_RAG},
        {"role": "user", "content": f"Context:\n{format_context(chunks)}\n\nQuestion: {question}"},
    ]

In [0]:
def ask_llm(messages: list[dict]) -> dict:
    response = llm_client.chat.completions.create(
        model=llm_cfg["model"],
        messages=messages,
        max_tokens=llm_cfg["max_tokens"],
        temperature=llm_cfg["temperature"],
    )
    return {
        "answer": response.choices[0].message.content,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
    }

In [0]:
import json
import time
import uuid
from datetime import datetime, timezone

from pyspark.sql.types import (
    ArrayType, BooleanType, DoubleType, IntegerType,
    StringType, StructField, StructType, TimestampType,
)

QA_LOG_SCHEMA = StructType([
    StructField("request_id", StringType(), False),
    StructField("asked_at", TimestampType(), False),
    StructField("caller", StringType(), False),
    StructField("question", StringType(), False),
    StructField("use_retrieval", BooleanType(), False),
    StructField("filters", StringType(), True),
    StructField("answer", StringType(), True),
    StructField("source_chunk_ids", ArrayType(StringType()), True),
    StructField("source_scores", ArrayType(DoubleType()), True),
    StructField("model", StringType(), False),
    StructField("prompt_tokens", IntegerType(), True),
    StructField("completion_tokens", IntegerType(), True),
    StructField("latency_ms", IntegerType(), False),
    StructField("status", StringType(), False),
    StructField("error", StringType(), True),
])


def write_qa_log(record: dict) -> None:
    try:
        (
            spark.createDataFrame([record], schema=QA_LOG_SCHEMA)
            .write.mode("append")
            .saveAsTable(qa_logs_table)
        )
    except Exception as error:
        logger.warning("Could not write QA log for request %s: %s", record["request_id"], error)

In [0]:
def answer_question(question: str, use_retrieval: bool = True, top_k: int | None = None,
                    filters: dict | None = None, caller: str = "qa") -> dict:
    started = time.perf_counter()
    record = {
        "request_id": str(uuid.uuid4()),
        "asked_at": datetime.now(timezone.utc),
        "caller": caller,
        "question": question,
        "use_retrieval": use_retrieval,
        "filters": json.dumps(filters) if filters else None,
        "answer": None,
        "source_chunk_ids": None,
        "source_scores": None,
        "model": llm_cfg["model"],
        "prompt_tokens": None,
        "completion_tokens": None,
        "status": "ok",
        "error": None,
    }

    try:
        chunks = retrieve(question, top_k=top_k, filters=filters) if use_retrieval else None
        result = ask_llm(build_messages(question, chunks))

        result["request_id"] = record["request_id"]
        result["question"] = question
        result["use_retrieval"] = use_retrieval
        result["sources"] = [
            {"number": n, "chunk_id": c["chunk_id"], "title": c["title"],
             "channel": c["channel_title"], "score": round(c["score"], 3)}
            for n, c in enumerate(chunks or [], start=1)
        ]

        record["answer"] = result["answer"]
        record["source_chunk_ids"] = [s["chunk_id"] for s in result["sources"]]
        record["source_scores"] = [float(s["score"]) for s in result["sources"]]
        record["prompt_tokens"] = result["prompt_tokens"]
        record["completion_tokens"] = result["completion_tokens"]
        return result

    except Exception as error:
        record["status"] = "error"
        record["error"] = str(error)[:1000]
        raise

    finally:
        record["latency_ms"] = int((time.perf_counter() - started) * 1000)
        write_qa_log(record)